# 1. Data Structure & Stability Analysis

**Objective**: Analyze the low-dimensional structure (PCA) and stability (CCA) of the neural population dynamics.

**Steps**:
1.  Load Preprocessed Data (`average_response_matrix_total`).
2.  **PCA**: Reduce dimensionality to 3D and visualize trajectories.
3.  **CCA**: Assess stability across trials (Split-Half Analysis).
4.  **Multi-Probe Analysis**: Compare dynamics across different brain regions.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cross_decomposition import CCA
from allensdk.brain_observatory.ecephys.visual_behavior_neuropixels_project_cache import VisualBehaviorNeuropixelsProjectCache

# Import Architectural Style
import viz_style
viz_style.apply_style()

# Load Data
session_id = 715093703
input_file = Path(f"../Dataset/Processed/{session_id}/average_response_matrix.pkl")

if input_file.exists():
    average_response_matrix_total = pd.read_pickle(input_file)
    print(f"Loaded data from {input_file}")
    print(f"Shape: {average_response_matrix_total.shape}")
else:
    print(f"Error: Data file not found at {input_file}")
    print("Please run '0_Data_Denoise.ipynb' first.")

# Need session object for CCA (spike counts)
# We'll reload it quickly
base_dir = "../Dataset"
cache = VisualBehaviorNeuropixelsProjectCache.from_local_cache(cache_dir=base_dir, use_static_cache=True)
session = cache.get_ecephys_session(session_id)

### Step 2: Linear Dimensionality Reduction (PCA)
Apply PCA to reduce the high-dimensional neural activity into 3 principal components for initial visualization.

In [ ]:
X = average_response_matrix_total.values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

pca = PCA(n_components=3)
X_pca = pca.fit_transform(X_scaled)

print(f"Original Shape: {X.shape}")
print(f"Reduced Shape: {X_pca.shape}")
print(f"Explained Variance Ratio: {pca.explained_variance_ratio_} (Total: {sum(pca.explained_variance_ratio_):.2%})")

# 3D Visualization
fig = plt.figure(figsize=(12, 10))
ax = fig.add_subplot(111, projection='3d')

# Use helper function
sc = viz_style.plot_3d_trajectory(
    ax,
    X_pca[:, 0], X_pca[:, 1], X_pca[:, 2],
    c=np.arange(len(X_pca)), # Color by time
    cmap='inferno',
    title="NEURAL POPULATION TRAJECTORY (PCA)",
    xlabel=f'PC1 ({pca.explained_variance_ratio_[0]:.2%})',
    ylabel=f'PC2 ({pca.explained_variance_ratio_[1]:.2%})',
    zlabel=f'PC3 ({pca.explained_variance_ratio_[2]:.2%})'
)

# Annotations
ax.text(X_pca[0, 0], X_pca[0, 1], X_pca[0, 2], "START", color=viz_style.GREEN, fontsize=10, fontweight='bold')
ax.text(X_pca[-1, 0], X_pca[-1, 1], X_pca[-1, 2], "END", color=viz_style.RED, fontsize=10, fontweight='bold')

plt.colorbar(sc, label='Time (Frame)')
plt.tight_layout()
plt.show()

### Step 3: Latent Variable Stability Analysis (CCA)

Instead of simply averaging all trials, we analyze the **stability** of the neural population dynamics across repeats.
We use **Canonical Correlation Analysis (CCA)** to compare the latent trajectories of two independent subsets of trials (e.g., First 5 repeats vs. Last 5 repeats).

High canonical correlations indicate that the "neural manifold" is stable and consistent across trials.

In [ ]:
def analyze_latent_stability(session, unit_ids, stimulus_name="natural_movie_one", n_components_pca=10, n_components_cca=3):
    """
    Analyzes the stability of neural representations across trials using PCA + CCA.
    """
    # 1. Get Stimulus Table
    stim_table = session.get_stimulus_table(stimulus_name)
    
    # 2. Get Spike Counts (Presentations x Neurons)
    frame_duration = stim_table['duration'].mean()
    
    print(f"Fetching spike counts for {len(unit_ids)} units...")
    spike_counts = session.presentationwise_spike_counts(
        stimulus_presentation_ids=stim_table.index.values,
        bin_edges=np.array([0, frame_duration]),
        unit_ids=unit_ids
    )
    
    # Sum spikes in bin (Frames x Neurons)
    X_all = spike_counts.sum(dim="time_relative_to_stimulus_onset").values
    
    # Reshape to (Repeats, Frames, Neurons)
    n_frames = 900
    n_repeats = len(stim_table) // n_frames
    n_units = X_all.shape[1]
    
    print(f"Reshaping data: {n_repeats} repeats x {n_frames} frames x {n_units} units")
    X_reshaped = X_all.reshape(n_repeats, n_frames, n_units)
    
    # 3. Split Data (First 5 vs Last 5)
    half = n_repeats // 2
    X_A = X_reshaped[:half].mean(axis=0) # Average of first half
    X_B = X_reshaped[half:].mean(axis=0) # Average of second half
    
    # 4. PCA Reduction (Denoising)
    pca_A = PCA(n_components=n_components_pca)
    pca_B = PCA(n_components=n_components_pca)
    
    P_A = pca_A.fit_transform(StandardScaler().fit_transform(X_A))
    P_B = pca_B.fit_transform(StandardScaler().fit_transform(X_B))
    
    # 5. CCA Alignment
    cca = CCA(n_components=n_components_cca)
    cca.fit(P_A, P_B)
    
    U_c, V_c = cca.transform(P_A, P_B)
    
    # Calculate correlations for each component
    corrs = [np.corrcoef(U_c[:, i], V_c[:, i])[0, 1] for i in range(n_components_cca)]
    
    return corrs, U_c, V_c

# Run Analysis for VISp units
visp_units = session.units[session.units.ecephys_structure_acronym == 'VISp']
print(f"Analyzing stability for Session {session_id} (VISp)...")
corrs, U_c, V_c = analyze_latent_stability(session, visp_units.index.values)

print("\n--- CCA Stability Results ---")
for i, r in enumerate(corrs):
    print(f"Canonical Correlation {i+1}: {r:.4f}")

In [ ]:
# Visualization of Stability
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# 1. Bar Plot of Correlations
axes[0].bar(range(1, len(corrs)+1), corrs, color='skyblue', edgecolor='black')
axes[0].set_ylim(0, 1.1)
axes[0].set_xlabel("Canonical Component")
axes[0].set_ylabel("Correlation Coefficient")
axes[0].set_title(f"Latent Variable Stability (Split-Half CCA)\nSession {session_id}")
axes[0].grid(axis='y', linestyle='--', alpha=0.7)

# 2. Trajectory Overlay (First Canonical Component)
time_axis = np.arange(900)
axes[1].plot(time_axis, U_c[:, 0], label="Split A (First Half)", color='blue', alpha=0.7)
axes[1].plot(time_axis, V_c[:, 0], label="Split B (Second Half)", color='orange', alpha=0.7, linestyle='--')
axes[1].set_xlabel("Frame (Time)")
axes[1].set_ylabel("Latent Activity (CC1)")
axes[1].set_title(f"Aligned Latent Trajectories (CC1, r={corrs[0]:.2f})")
axes[1].legend()

plt.tight_layout()
plt.show()

### Step 4: Multi-Probe Analysis
Compare dynamics across different brain regions (Inter-Probe Similarity) and assess stability within each probe (Intra-Probe Stability).

In [ ]:
import seaborn as sns

# 1. Identify Probes and their Units
probe_units_dict = {}
for probe_id, row in session.probes.iterrows():
    probe_name = row['description']
    units = session.units[session.units.probe_id == probe_id]
    if len(units) > 10:
        probe_units_dict[probe_name] = units.index.values
        print(f"Probe {probe_name}: {len(units)} units")

probe_names = sorted(probe_units_dict.keys())

# --- Analysis 1: Intra-Probe Stability ---
print("\n--- 1. Intra-Probe Stability (Split-Half CCA) ---")
stability_scores = {}

for name in probe_names:
    units = probe_units_dict[name]
    n_comp = min(10, len(units))
    try:
        corrs, _, _ = analyze_latent_stability(session, units, n_components_pca=n_comp, n_components_cca=3)
        stability_scores[name] = corrs[0]
        print(f"{name}: CC1 = {corrs[0]:.4f}")
    except Exception as e:
        print(f"{name}: Failed ({e})")
        stability_scores[name] = 0

# --- Analysis 2: Inter-Probe Similarity ---
print("\n--- 2. Inter-Probe Similarity (Pairwise CCA) ---")
n_probes = len(probe_names)
similarity_matrix = np.zeros((n_probes, n_probes))

probe_trajectories = {}
for name in probe_names:
    units = probe_units_dict[name]
    valid_units = [u for u in units if u in average_response_matrix_total.columns]
    if len(valid_units) < 5:
        probe_trajectories[name] = None
        continue
    X_probe = average_response_matrix_total[valid_units].values
    n_comp = min(10, len(valid_units))
    pca = PCA(n_components=n_comp)
    P_probe = pca.fit_transform(StandardScaler().fit_transform(X_probe))
    probe_trajectories[name] = P_probe

cca = CCA(n_components=1)
for i, name_i in enumerate(probe_names):
    for j, name_j in enumerate(probe_names):
        if i == j:
            similarity_matrix[i, j] = 1.0
            continue
        traj_i = probe_trajectories[name_i]
        traj_j = probe_trajectories[name_j]
        if traj_i is None or traj_j is None:
            continue
        cca.fit(traj_i, traj_j)
        U, V = cca.transform(traj_i, traj_j)
        corr = np.corrcoef(U[:, 0], V[:, 0])[0, 1]
        similarity_matrix[i, j] = corr

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(18, 7))
axes[0].bar(stability_scores.keys(), stability_scores.values(), color='teal', alpha=0.7)
axes[0].set_ylim(0, 1.0)
axes[0].axhline(0.8, color='red', linestyle='--', linewidth=1, label='High Stability Threshold (0.8)')
axes[0].set_title("Intra-Probe Stability")
axes[0].set_ylabel("Canonical Correlation (CC1)")
axes[0].legend()

sns.heatmap(similarity_matrix, annot=True, fmt=".2f", cmap="magma", 
            xticklabels=probe_names, yticklabels=probe_names, vmin=0, vmax=1, ax=axes[1])
axes[1].set_title("Inter-Probe Similarity")
plt.tight_layout()
plt.show()